In [1]:
# Elizabeth Van Der Schaaf
# US Housing Market Dashboard
# 2/9/2025

# I attempted to recreate a dashboard I designed in Tableau with a few modifications.
# Links between the two dashbaords are displayed prominently at the top.
# Labels were left off of the average home value by region map to reduce clutter.
# A slider was added to the price drop dashboard so that the data can now be filtered by year.

import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.graph_objects as go
import pandas as pd

# Load data
data = pd.read_csv("housing_trends_data.csv")
key_metrics = ['CutRaw', 'DaysPending', 'HomeValue']
filtered_data = data[data['measure_name'].isin(key_metrics)]

# Group by Year, State, and measure_name to get the average per year
yearly_data = filtered_data.groupby(['Year', 'State', 'measure_name'])['measure_value'].mean().reset_index()

# Pivot data so each measure has a column per year
yearly_data = yearly_data.pivot_table(index=['State', 'measure_name'], columns='Year', values='measure_value')

# Flatten the multi-level columns (Year as columns) to make it more readable
yearly_data.columns = [f'{col}' for col in yearly_data.columns] 

# Reset index for better readability
yearly_data.reset_index(inplace=True)

# Rename 'CutRaw' to 'PriceCut'
yearly_data['measure_name'] = yearly_data['measure_name'].replace('CutRaw', 'PriceCut')

# Extract data for HomeValue and PriceCut
home_value_data = yearly_data[yearly_data['measure_name'] == 'HomeValue']
price_drop_data = yearly_data[yearly_data['measure_name'] == 'PriceCut']
days_data = yearly_data[yearly_data['measure_name'] == 'DaysPending']
years = home_value_data.columns[2:].tolist()

# -----------------
# Create the Dash app
app = dash.Dash(__name__, suppress_callback_exceptions=True)

# Create choropleth map for Price Drop
pricedrop_map = go.Figure(go.Choropleth(
    locations=price_drop_data['State'],  # States as locations
    locationmode='USA-states',           # USA states mode
    z=price_drop_data['2018'],           # Using the 2018 data
    hoverinfo='location+z',              # Hover text shows the state and value
    colorscale='Oranges',            
    colorbar_title='Price Drop', 
))

# Update geo settings to focus on the U.S.
pricedrop_map.update_geos(
    projection_type='albers usa',        # Projection type for the U.S.
    showland=True,                       # Show land
    landcolor='rgb(255, 255, 255)',      # Land color
    subunitcolor='rgb(255, 255, 255)',   # State borders color
)

pricedrop_map.update_layout(
    title='Average Drop in Price',
)

# Create choropleth map for Days Pending
days_map = go.Figure(go.Choropleth(
    locations=days_data['State'],        # States as locations
    locationmode='USA-states',           # USA states mode
    z=days_data['2018'],                 # Using the 2018 data
    hoverinfo='location+z',              # Hover text shows the state and value
    colorscale='Greys',               
    colorbar_title='Days on Market', 
))

# Update geo settings to focus on the U.S.
days_map.update_geos(
    projection_type='albers usa',        # Projection type for the U.S.
    showland=True,                       # Show land
    landcolor='rgb(255, 255, 255)',      # Land color
    subunitcolor='rgb(255, 255, 255)',   # State borders color
)

# Create combo chart for Days Pending (Bar) and Price Drop (Line)
daysdrop_combo = go.Figure()

# Add a bar chart for Days Pending
daysdrop_combo.add_trace(go.Bar(
        x=years,  # Use years for x-axis
        y=days_data[years].iloc[0],  # Use the selected year data for the Y-axis
        name='Days on Market',
        marker_color='lightgrey',  # Bar color
        yaxis='y1',  # Link this trace to the first y-axis
))

# Add a line chart for Price Drop
daysdrop_combo.add_trace(go.Scatter(
        x=years,  # Use years for x-axis
        y=price_drop_data[years].iloc[0],  # Use the selected year data for the Y-axis
        mode='lines+markers',
        name='Price Drop',
        line=dict(color='orange', width=3),
        marker=dict(size=8, color='orange'),
        yaxis='y2',  # Link this trace to the second y-axis
))

# Update layout for combo chart
daysdrop_combo.update_layout(
        title=f'Larger price cuts and faster sales',
        xaxis_title='Year',
        yaxis_title='Days on Market',
        yaxis2=dict(
            title='Price Drop',
            overlaying='y',  # Overlay this axis on the same plot
            side='right',  # Place the second y-axis on the right
        ),
        barmode='group',  # Group bars together
        height=600,
)

# Get a list of years present in the data
years = price_drop_data.columns[2:].tolist()

# Define layouts for each dash
dash_1_layout = html.Div([
    html.H1("US Housing Market | How does a drop in home price affect time to sell?"),
    html.P("Click on the average price drop map to filter by state."),
        # Slider for year selection
    dcc.Slider(
        id='year-slider',
        min=int(min(years)), 
        max=int(max(years)), 
        step=1,  # 1 year at a time
        marks={int(year): str(year) for year in years},  # Create marks based on the years
        value=int(min(years)),  # Default value to the first year in the dataset
    ),

        # Container for both maps side by side
    html.Div([
        # Price Drop map
        html.Div(
            dcc.Graph(id='price-drop-map', figure=pricedrop_map),
            style={'width': '50%', 'display': 'inline-block'} 
        ),
                
        # Days Pending map
        html.Div(dcc.Graph(id='days-map', figure=days_map),
            style={'width': '50%', 'display': 'inline-block'}
        ),
    ], style={'display': 'flex'}),  # display in a row
    
    # Combo Chart (Bar for Days Pending and Line for Price Drop)
    html.Div([
        dcc.Graph(id='daysdrop-combo', figure=daysdrop_combo),
    ], style={'marginTop': '30px'}),  # Add margin to separate from the maps
])

# Define callback to update both maps based on selected year
@app.callback(
    [Output('price-drop-map', 'figure'),
     Output('days-map', 'figure')],
    [Input('year-slider', 'value')]  # Input for year slider
)
def update_maps(selected_year):
    selected_year = str(selected_year)

    # Filter the data for selected year
    price_drop_values = price_drop_data[selected_year].values
    days_values = days_data[selected_year].values

    # Create updated Price Drop map
    price_drop_fig = go.Figure(go.Choropleth(
        locations=price_drop_data['State'],
        locationmode='USA-states',
        z=price_drop_values,
        hoverinfo='location+z',
        colorscale='Oranges',
        colorbar_title='Price Drop',
        ))
    # Update geo settings for Price Drop map
    price_drop_fig.update_geos(
        projection_type='albers usa',
        showland=True,
        landcolor='rgb(255, 255, 255)',
        subunitcolor='rgb(255, 255, 255)',
)
    
    pricedrop_map.update_layout(
        title='Average Drop in Price',
)
    
    # Create updated Days Pending map
    days_fig = go.Figure(go.Choropleth(
        locations=days_data['State'],
        locationmode='USA-states',
        z=days_values,
        hoverinfo='location+z',
        colorscale='Greys',
        colorbar_title='Days on Market',
        ))

    # Update geo settings for Days Pending map
    days_fig.update_geos(
        projection_type='albers usa',
        showland=True,
        landcolor='rgb(255, 255, 255)',
        subunitcolor='rgb(255, 255, 255)',
    )

    # Return the updated figures for both maps
    return price_drop_fig, days_fig

# Define the callback to update the combo chart based on selected state from the Price Drop map
@app.callback(
    Output('daysdrop-combo', 'figure'), 
    Input('price-drop-map', 'clickData'), 
    [Input('year-slider', 'value')] 
)
def update_daysdrop_combo(clickData, selected_year):
    if clickData is None:
        # Initialize the figure for default display
        daysdrop_combo = go.Figure()
        daysdrop_combo.update_layout(
            title=f'Larger price cuts and faster sales',
            xaxis_title='Year',
            yaxis_title='Days on Market',
            yaxis2=dict(
                title='Price Drop',
                overlaying='y',  # Overlay this axis on the same plot
                side='right',  # Place second y-axis on the right
        ),
            barmode='group',  # Group bars together
            height=600,
)
        return daysdrop_combo

    # Get selected state from the map clickData
    selected_state = clickData['points'][0]['location']  # State name from the map click

    # Filter data for selected state and selected year
    state_price_drop_data = price_drop_data[price_drop_data['State'] == selected_state]
    state_days_data = days_data[days_data['State'] == selected_state]

    # Convert selected_year to string to match column names in data
    selected_year = str(selected_year)

    # Create a new combo chart based on the selected state
    daysdrop_combo = go.Figure()

    # Add a bar chart for Time on Market with a secondary y-axis
    daysdrop_combo.add_trace(go.Bar(
        x=years, 
        y=state_days_data[years].iloc[0], 
        name='Days on Market',
        marker_color='grey', 
        yaxis='y1',  # Link this trace to the first y-axis
))
  
    # Add a line chart for Price Drop with a different y-axis
    daysdrop_combo.add_trace(go.Scatter(
        x=years, 
        y=state_price_drop_data[years].iloc[0], 
        mode='lines+markers',
        name='Price Drop',
        line=dict(color='orange', width=3),
        marker=dict(size=8, color='orange'),
        yaxis='y2',  # Link this trace to the second y-axis
))
    # Update layout for combo chart
    daysdrop_combo.update_layout(
        title=f'Larger price cuts and faster sales - Relationship between price cuts and time on market for {selected_state}',
        xaxis_title='Year',
        yaxis_title='Days on Market',
        yaxis2=dict(
            title='Price Drop',
            overlaying='y',  # Overlay this axis on the same plot
            side='right',  # Place the second y-axis on the right
),
        barmode='group',  # Group bars together
        height=600,
)
    return daysdrop_combo


#-------------------
    # Create choropleth map for Home Values
choropleth_map = go.Figure(go.Choropleth(
    locations=home_value_data['State'],  # States as locations
    locationmode='USA-states',           # USA states mode
    z=home_value_data['2018'],           # Using the 2023 data (or any year you want)
    hoverinfo='location+z',              # Hover text shows the state and value
    colorscale='Purples',                # Use a color scale (e.g., Viridis)
    colorbar_title='Home Value',         # Color bar title
))

# Update geo settings to focus on the U.S.
choropleth_map.update_geos(
    projection_type='albers usa',        # Projection type for the U.S.
    showland=True,                       # Show land
    landcolor='rgb(255, 255, 255)',      # Land color
    subunitcolor='rgb(255, 255, 255)',   # State borders color
)

# Update layout with title
choropleth_map.update_layout(
    title='Average home prices vary greatly by region',
    geo=dict(
        projection_type='albers usa',    # Set projection to focus on the U.S.
        showland=True,                   # Show land
        landcolor='rgb(255, 255, 255)',  # Land color
        subunitcolor='rgb(255, 255, 255)', # Border color for states
),
)
# Create dropdown buttons dynamically
years = home_value_data.columns[2:].tolist()
buttons = []
for year in years:
    buttons.append({
        'args': [{'z': [home_value_data[year].values]}],  # Update z values for the specific year
        'label': year,                                   # Label for each year
        'method': 'restyle'                               # Use 'restyle' to update data
})

# Update layout with title and dropdown menu
choropleth_map.update_layout(
    title='Average home prices vary greatly by region',
    geo=dict(
        projection_type='albers usa',    # Set projection to focus on the U.S.
        showland=True,                   # Show land
        landcolor='rgb(255, 255, 255)',  # Land color
        subunitcolor='rgb(255, 255, 255)', # Border color for states
),
    updatemenus=[{
        'buttons': buttons,              # Use the dynamically created buttons list
        'direction': 'down',
        'pad': {'r': 10, 't': 10},        # Padding for dropdown
        'showactive': True,
        'x': 0.1,                        # Position of dropdown on the x-axis
        'xanchor': 'left',
        'y': 1.1,                        # Position of dropdown on the y-axis
        'yanchor': 'top',
    }],
)

# Create combo chart for HomeValue and PriceCut
combo_chart = go.Figure()

# Add Bar chart for Average Home Value
combo_chart.add_trace(go.Bar(
    x=home_value_data.columns[2:],  # Use years as the x-axis
    y=home_value_data.iloc[0, 2:],  # Default to the first state
    name='Average Home Value',
    yaxis='y1',  # Left y-axis
    marker=dict(color='rgb(190,178,213)')
))

# Add Line chart for Average Price Cut
combo_chart.add_trace(go.Scatter(
    x=price_drop_data.columns[2:],  # Use years as the x-axis
    y=price_drop_data.iloc[0, 2:],  # Default to the first state
    name='Average Price Cut',
    mode='lines+markers',
    yaxis='y2',  # Right y-axis
    line=dict(color='orange', width=3)
))

# Update layout for combo chart
combo_chart.update_layout(
    title='Average Home Sales and Price Cut by Year',
    xaxis=dict(
        title='Year',
        tickmode='array',
        tickvals=home_value_data.columns[2:],  # Use years as ticks
),
    yaxis=dict(
        title='Average Home Value',
        titlefont=dict(color='grey'),
        tickfont=dict(color='grey'),
),
    yaxis2=dict(
        title='Average Price Cut',
        titlefont=dict(color='grey'),
        tickfont=dict(color='grey'),
        overlaying='y',  # Overlay the y-axes
        side='right',    # Place second y-axis on the right
),
)

# ------------------------
dash_2_layout = html.Div([
    html.H1("US Housing Market | Do price cuts change with home value?"),
    html.P("Click on the map to filter by state."),
    dcc.Graph(id='average-value-map', figure=choropleth_map), 
    dcc.Graph(id='combo-chart', figure=combo_chart)
])

# Define the callback to update combo chart based on selected state
@app.callback(
    Output('combo-chart', 'figure'),
    Input('average-value-map', 'clickData')
)
def update_graph(clickData):
    # Check if a state has been selected on the map
    if clickData:
        selected_state = clickData['points'][0]['location'] 

        # Filter data for the selected state
        selected_home_value = home_value_data[home_value_data['State'] == selected_state].iloc[0, 2:]
        selected_price_drop = price_drop_data[price_drop_data['State'] == selected_state].iloc[0, 2:]

        # Create updated combo chart for the selected state
        fig = go.Figure()

        # Bar chart for Average Home Sales (HomeValue)
        fig.add_trace(go.Bar(
            x=home_value_data.columns[2:],  # Use years as the x-axis
            y=selected_home_value,  # Data for the selected state
            name='Average Home Value',
            yaxis='y1',  # Left y-axis
            marker=dict(color='rgb(190,178,213)')
))

        # Line chart for Average Price Cut (PriceCut)
        fig.add_trace(go.Scatter(
            x=price_drop_data.columns[2:],  # Use the years as the x-axis
            y=selected_price_drop,  # Data for the selected state
            name='Average Price Cut',
            mode='lines+markers',
            yaxis='y2',  # Right y-axis
            line=dict(color='orange', width=3)
))
                        
        # Update layout
        fig.update_layout(
            title=f'As home prices rise price cuts tend to follow',
            xaxis=dict(
                title='Year',
                tickmode='array',
                tickvals=home_value_data.columns[2:],  # Use years as ticks
),
            yaxis=dict(
                title='Average Home Value',
                titlefont=dict(color='grey'),
                tickfont=dict(color='grey'),
),
            yaxis2=dict(
                title='Average Price Cut',
                titlefont=dict(color='grey'),
                tickfont=dict(color='grey'),
                overlaying='y',  # Overlay the y-axes
                side='right',    # Place the second y-axis on the right
),
)
    else:
        # If no state is selected, show data for the first state
        fig = combo_chart

    return fig
    
#-------------------
# Define main layout with navigation links and page content
app.layout = html.Div([
    dcc.Location(id='url', refresh=False),  # Location component to track the URL
    html.Div([
        dcc.Link('Go to Price Drop Dashboard', href='/page-1'),
        html.Br(),
        dcc.Link('Go to Average Home Value Dashboard', href='/page-2'),
    ]),
    html.Div(id='page-content'),  # This will update based on URL path
])

# Callback to switch between pages based on URL
@app.callback(
    Output('page-content', 'children'),
    [Input('url', 'pathname')]
)
def display_page(pathname):
    if pathname == '/page-1':
        return dash_1_layout  # Show the content of Price Drop Dashboard
    elif pathname == '/page-2':
        return dash_2_layout  # Show the content of Average Home Value Dashboard
    else:
        return dash_1_layout  # Default to Price Drop Dashboard if the URL path is not recognized

if __name__ == '__main__':
    app.run(debug=True, jupyter_mode="tab", use_reloader=False)
    #app.run(juypter_mode="tab")
    #app.run_server(juypter_mode="tab", port=8051)



Dash app running on http://127.0.0.1:8050/


<IPython.core.display.Javascript object>